## JUnit
JUnit is the de facto Java unit testing framework. While JUnit 5 (Jupiter) is the modern standard, JUnit 4 is widely found in legacy codebases and is discussed here. Before actually using JUnit, let us write our own testing logic. We'll be testing the below Calculator class:

In [ ]:
class Calculator {
    public int add(int x, int y){
        return x + y;
    }
    
    public int subtract(int x, int y){
        return x - y;
    }

    public int divide(int x, int y){
        return x / y;
    }
}

To test the functionality, we can write the following:

In [ ]:
class CalculatorTest {
    public static void main(String... args){
        Calculator calc = new Calculator();
        
        int result = calc.add(5, 6);
        int expected = 11;
        
        if(result != expected){
            System.out.println("Unexpected result " + result);
        }
    }
}

It would be better if we use an exception to denote test failure. Meanwhile it would be better if we modularize our code, such that testing other aspects of calculator becomes a bit easier.

In [ ]:
class CalculatorTest {
    public int errorCount = 0;
    
    public void testAdd(){
        Calculator calc = new Calculator();
        
        int result = calc.add(5, 6);
        int expected = 11;
        
        if(result != expected){
            throw new IllegalStateException("Unexpected result " + result);
        }
    }
    
    public static void main(String... args){
        CalculatorTest cTest = new CalculatorTest();
        
        try {
            cTest.testAdd();
        } catch (Exception e){
            cTest.errorCount++;
            e.printStackTrace();
        }
        
        if(cTest.errorCount > 0){
            throw new IllegalStateException("There were " + cTest.errorCount + " errors");
        }
    }
}

Using a testing framework like JUnit removes some of the extra code that we have to write. We would just need to define test cases; it is the responsibility of the framework to execute them and present the result. Given that, we need to familiarize ourselves with some of the associated terms:
- **Test:** a method annotated with `@Test`
- **Test class:** contains one or many tests
- **Suite:** allows us to group test classes
- **Runner:** class that runs tests.

The equivalent JUnit test class is:

In [ ]:
import static org.junit.Assert.assertEquals;

import org.junit.Test;

public class CalculatorTest {
    
    @Test
    public void testAdd() {
        Calculator calculator = new Calculator();
        int result = calculator.add(5, 6);
        int expected = 11;

        // JUnit 4 signature: assertEquals(message, expected, actual)
        assertEquals("Unexpected result", expected, result);
    }
}

There are a number of assertions available:

In [ ]:
import static org.junit.Assert.*;
import org.junit.Test;

@Test
public void testAssertions() {
    // Uses equals
    assertEquals("Hello", new String("Hello")); // Pass
    assertEquals("Hello", "Hello");             // Pass

    // Uses ==
    assertSame("Hello", new String("Hello")); // Fail
    assertSame("Hello", "Hello");             // Pass

    // Tests for content of arrays
    assertArrayEquals(new char[] { 'A', 'B', 'C' }, 
                      "ABC".toCharArray());     // Pass
    assertArrayEquals(new String[] { new String("H"), "E", "O" }, 
                      new String[] { "H", "E", "O" }); // Pass
    
    // True / False
    assertTrue(5 == 5);    // Pass
    assertFalse(5 > 6);    // Pass

    assertNull(null);      // Pass
}

There are also negations of the above assertions available such as `assertNotNull` and `assertNotSame`.  

If some code needs to be repeated before and after every test execution, we can make use of `@Before` and `@After` annotations:

In [ ]:
import static org.junit.Assert.assertEquals;
import org.junit.After;
import org.junit.Before;
import org.junit.Test;

public class CalculatorTest {
    private Calculator calculator;

    @Before
    public void initCalculator() {
        calculator = new Calculator();
    }

    @After
    public void destroyCalculator() {
        calculator = null;
    }

    @Test
    public void testAdd() {
        int result = calculator.add(5, 6);
        int expected = 11;

        assertEquals("Unexpected result", expected, result);
    }

    @Test
    public void testSubtract() {
        int result = calculator.subtract(5, 6);
        int expected = -1;

        assertEquals("Unexpected result", expected, result);
    }
}

There are also `@BeforeClass` and `@AfterClass` annotations (applied to `public static` methods). These methods run before and after all tests in the class have completed.  

To test for the occurrence of an exception:

In [ ]:
@Test(expected = ArithmeticException.class)
public void testDivisionByZero() {
    calculator.divide(5, 0);
}

To test for timeout (specified in milliseconds):

In [ ]:
import org.junit.Test;

@Test(timeout = 10) // Timeout in milliseconds (10 ms)
public void testTimeout() {
    // Long running task
}

To exclude a test from execution:

In [ ]:
import org.junit.Ignore;
import org.junit.Test;

@Test
@Ignore("Skipped until bug #123 is resolved")
public void someTestToBeSkipped(){
    // Implementation
}

**Runner:** JUnit by default uses `BlockJUnit4ClassRunner` to run all tests. We can use the `@RunWith` annotation to use a custom runner. The `Suite` runner helps group test classes together and run them as a test suite:

In [ ]:
import org.junit.runner.RunWith;
import org.junit.runners.Suite;
import org.junit.runners.Suite.SuiteClasses;

@RunWith(Suite.class)
@SuiteClasses(value = { CalculatorTest.class, ConverterTest.class })
public class MachineTestSuite {

}

There is another runner called the parameterized runner. It helps test for multiple values in a single test.

In [ ]:
import java.util.Arrays;
import java.util.Collection;
import static org.junit.Assert.assertEquals;
import org.junit.Test;
import org.junit.runner.RunWith;
import org.junit.runners.Parameterized;
import org.junit.runners.Parameterized.Parameter;
import org.junit.runners.Parameterized.Parameters;

@RunWith(Parameterized.class)
public class CalculatorTest {

    @Parameter(1)
    public int operand1;
    @Parameter(2)
    public int operand2;
    @Parameter(0)
    public int expected;

    @Parameters(name="{index}: {1}+{2}={0}")  // Name parameter helps identify the test case
    public static Collection<Integer[]> data() {
        return Arrays.asList(new Integer[][] { 
            { 0, 0, 0 }, { 0, 5, -5 }, { 10, 6, 4 }, { -6, -3, -3 } // All the test cases
        });
    }

    @Test
    public void testAdd() {
        Calculator calculator = new Calculator();
        
        int result = calculator.add(operand1, operand2);
        assertEquals("Unexpected result", expected, result);
    }
}

### Assertions
JUnit comes with the `Assert` class which contains several static assert methods such as `assertTrue`, `assertFalse`, `assertEquals`, `assertArrayEquals`, `assertSame`, etc. These methods throw `AssertionError` on failure. These methods also have overloaded variants which accept a message passed to the `AssertionError` (`assertEquals(message, expected, actual)`).

Another way to write assertions is using the Hamcrest `assertThat` method in combination with Hamcrest matchers. JUnit itself has an `Assert.assertThat` method which is more or less the same as the Hamcrest one, but since JUnit 4.13, it has been deprecated in favor of `org.hamcrest.MatcherAssert.assertThat`. Note that richer matchers (e.g. `equalToIgnoringCase`, `hasSize`, `contains`) require the `hamcrest-library` dependency beyond JUnit's bundled `hamcrest-core`. For example:

In [ ]:
import static org.hamcrest.MatcherAssert.assertThat;
import static org.hamcrest.Matchers.equalToIgnoringCase;

assertThat("hello", equalToIgnoringCase("HELLO"));

Some other examples:

In [ ]:
import static org.hamcrest.MatcherAssert.assertThat;
import org.hamcrest.Matchers;
import java.util.ArrayList;
import java.util.List;

List<Integer> nums = new ArrayList<>();
nums.add(5); nums.add(6);
assertThat(nums, Matchers.hasSize(2));
assertThat(nums, Matchers.contains(5, 6));

String[] items = {"a", "b", "c"};
assertThat(items, Matchers.arrayWithSize(3));
assertThat(items, Matchers.arrayContaining("a", "b", "c"));
assertThat(items, Matchers.hasItemInArray("a"));

assertThat(1.2, Matchers.closeTo(1.0, 0.5));
assertThat("", Matchers.isEmptyOrNullString());

We can improve readability using the `is` matcher:

In [ ]:
import static org.hamcrest.MatcherAssert.assertThat;
import static org.hamcrest.Matchers.*;

assertThat(" World", is(equalToIgnoringWhiteSpace("World")));

To negate, use `not`:

In [ ]:
import static org.hamcrest.MatcherAssert.assertThat;
import static org.hamcrest.Matchers.*;

String str1 = "calligraphy";
String str2 = "painting";
// Pass: "calligraphy" does not contain "painting"
assertThat(str1, not(containsString(str2)));

Logical OR equivalent (`anyOf`):

In [ ]:
import static org.hamcrest.MatcherAssert.assertThat;
import static org.hamcrest.Matchers.*;

String str = "calligraphy";
String start = "call";
String end = "foo";
assertThat(str, anyOf(startsWith(start), containsString(end)));

Logical AND equivalent (`allOf`):

In [ ]:
import static org.hamcrest.MatcherAssert.assertThat;
import static org.hamcrest.Matchers.*;

String str = "calligraphy";
String start = "call";
String end = "phy";
assertThat(str, allOf(startsWith(start), endsWith(end)));

`AssertJ` is another library that provides fluent assertion capabilities (e.g. `assertThat(str).startsWith("call").endsWith("phy");`). A comparison with Hamcrest is illustrated [here](https://dzone.com/articles/hamcrest-vs-assertj-assertion-frameworks-which-one). Unlike standard JUnit 4 and Hamcrest, `AssertJ` provides **soft assertions** (`SoftAssertions`), which allow all failing assertions to be collected and reported together rather than failing at the very first assertion failure. (Note: JUnit 5 also natively supports grouped assertions via `assertAll()`).

## Mockito
Mockito mocks not only interfaces but also abstract classes and concrete classes. 

> **Note:** In modern Mockito (version 5.0+), the `mock-maker-inline` is enabled by default, allowing `final` classes and methods to be mocked out of the box. Additionally, since Mockito 3.4.0+, static methods can be mocked directly using `Mockito.mockStatic(...)`. Methods `equals()` and `hashCode()` cannot and should not be mocked.

As an example:

In [ ]:
import static org.junit.Assert.assertEquals;
import static org.mockito.Mockito.mock;
import org.junit.Test;

public class PrintService {
    public String print(String message) throws InterruptedException {
        Thread.sleep(1000);
        System.out.println("Message: " + message);
        return message;
    }
}

PrintService printService = mock(PrintService.class);

@Test
public void testPrintMessage() throws InterruptedException {
    String actual = printService.print("Hello World!");
    String expected = "Hello World!";

    assertEquals(expected, actual); // This assertion will fail, actual is null
}

By default, all methods of a mock return “uninitialized” or “empty” values, e.g.:
- `0` for numeric types (both primitive and boxed, like `int` and `Integer`)
- `false` for booleans (both `boolean` and `Boolean`)
- Empty collections/maps for `List`, `Set`, `Map`, etc.
- `Optional.empty()` for `Optional`
- `null` for most other reference types (including `String`)

We configure the mock and define what to do when specific methods are called. This is called **stubbing**.

In [ ]:
import static org.junit.Assert.assertEquals;
import static org.mockito.Mockito.when;
import org.junit.Test;

@Test
public void testPrintMessage() throws InterruptedException {
    // defining what the method does when called
    when(printService.print("Hello World!")).thenReturn("Hello World!");

    String actual = printService.print("Hello World!");
    String expected = "Hello World!";

    assertEquals(expected, actual);
}

Another way to express the same stubbing is using the `doReturn` method. This prefix style is especially useful (and required) when working with `spy` objects (to avoid executing the real method during stubbing) or when stubbing `void` methods:

In [ ]:
import static org.mockito.Mockito.doReturn;

doReturn("Hello World!").when(printService).print("Hello World!");

We can also specify multiple values which will be returned as the results of consecutive method calls. The last value will be used as a result for all further method calls.

In [ ]:
when(printService.print("Hello World!")).thenReturn("Hello World!", "Foo Bar");
// or
doReturn("Hello World!", "Foo Bar").when(printService).print("Hello World!");
// or
when(printService.print("Hello World!")).thenReturn("Hello World!").thenReturn("Foo Bar");

To set up a custom answer to be returned when a method is called, we do:

In [ ]:
import static org.mockito.Mockito.when;
import static org.mockito.Mockito.doAnswer;

when(printService.print("World!")).thenAnswer(invocation -> "Hello " + invocation.getArgument(0));
// or        
doAnswer(invocation -> "Hello " + invocation.getArgument(0)).when(printService).print("World!");

This can be used to throw exceptions:

In [ ]:
import static org.mockito.Mockito.when;

when(printService.print(null)).thenAnswer(invocation -> { throw new IllegalArgumentException(); });
// but prefer the cleaner syntax below:
when(printService.print(null)).thenThrow(IllegalArgumentException.class);
// or
when(printService.print(null)).thenThrow(new IllegalArgumentException("Invalid message"));

If for some reason we want to delegate a method call to the actual method (calling real implementation on a mock):

In [ ]:
import static org.junit.Assert.assertEquals;
import static org.mockito.Mockito.*;
import java.util.Date;

Date mock = mock(Date.class);
when(mock.getTime()).thenCallRealMethod();
doCallRealMethod().when(mock).setTime(42L);
mock.setTime(42L);
assertEquals(42L, mock.getTime());

Note that if we create a mock of an interface and try to configure a stub to call a real method, Mockito will throw an exception, as expected. Also, Mockito ensures that the exception being thrown is valid for that specific stubbed method and will complain if the exception is not in the method’s checked exceptions list.

Sometimes we may want to stub a method for a wide range of arguments (instead of a single value as shown earlier). To do this we make use of `ArgumentMatcher`:

In [ ]:
import static org.mockito.Mockito.*;
import static org.mockito.ArgumentMatchers.*;

public interface Uppercase {
    boolean close(String input);
}

Uppercase uC = mock(Uppercase.class);
when(uC.close(anyString())).thenReturn(true);

Mockito requires you to provide all arguments either by matchers or by exact values. If a method has more than one argument and we want to use argument matchers for only some of its arguments, it cannot be done directly. We cannot write code like this (will throw `InvalidUseOfMatchersException` at runtime):

In [ ]:
import static org.mockito.Mockito.*;
import static org.mockito.ArgumentMatchers.*;

// Consider the method
public interface Check {
    public boolean positive(int number, int delay);
}

Check check = mock(Check.class);
// WRONG: mixing raw value 5 and matcher anyInt() will fail at runtime
when(check.positive(5, anyInt())).thenReturn(true);

We need to wrap the literal value with the `eq(...)` matcher:

In [ ]:
import static org.mockito.Mockito.*;
import static org.mockito.ArgumentMatchers.*;

public interface Check {
    public boolean positive(int number, int delay);
}

Check check = mock(Check.class);
// eq ArgumentMatcher means specific value
// anyInt means any integer passed
when(check.positive(eq(5), anyInt())).thenReturn(true);

Some commonly used `ArgumentMatcher`s include `anyInt()`, `anyString()`, `anyList()`, `any(Class.class)`, `eq()`, `isNull()`, `isNotNull()`, etc.

To reset a mock so that all stubbed methods return back to their original default behavior, use `reset` (though in practice, creating fresh mocks per test with `@Before` is preferred):

In [ ]:
import static org.mockito.Mockito.*;
import static org.mockito.ArgumentMatchers.*;

Check check = mock(Check.class);
when(check.positive(anyInt(), anyInt())).thenReturn(true);

check.positive(5, 10); // returns true

reset(check);
check.positive(5, 10); // returns false (reset back to default)

### Verification
Mockito provides a way to inspect behaviour of mocks - whether a method was called, how many times, with what arguments, etc.

To verify that a method was called:

In [ ]:
List<Integer> numbers = mock(List.class);

// Verify that add was called with argument 1
verify(numbers).add(1);
// The above statemetn is short for
verify(numbers, times(1)).add(1);  // other options other than times are atMost(), atLeast()

// Verify that some method was not called
verify(numbers, never()).remove(anyInt());

We can also check that methods were called in specific order:

In [ ]:
List<Integer> numbers = mock(List.class);
numbers.add(1);
numbers.remove(1);
boolean present = numbers.contains(1);

// Below verification will fail as the actual order is different
InOrder order = inOrder(numbers);
order.verify(numbers).add(anyInt());
order.verify(numbers).contains(anyInt());
order.verify(numbers).remove(anyInt());

We can use `ArgumentCaptor` to capture arguments with which a certain method was called:

In [ ]:
// Unlike a mock, a spy calls all the original methods, unless we mock
// its specific methods
ArrayList<Integer> numbers = spy(new ArrayList<>());
numbers.addAll(List.of(1, 2, 3));
numbers.contains(2);

// Verify that calling contains(2) internally calls indexOf(2)
ArgumentCaptor<Integer> indexOfArgument = ArgumentCaptor.forClass(Integer.class);
verify(numbers).indexOf(indexOfArgument.capture());
Integer argumentValue = indexOfArgument.getValue();
assertThat(argumentValue, equalTo(2));

### Mockito Annotations
Mockito provides a number of annotations to declaratively supply mocks and dependencies, like `@Mock` and `@InjectMocks`. Consider the example below:

In [ ]:
import static org.junit.Assert.assertEquals;
import org.junit.Test;
import org.junit.runner.RunWith;
import org.mockito.InjectMocks;
import org.mockito.Mock;
import org.mockito.Mockito;
import org.mockito.junit.MockitoJUnitRunner;

// Need to test Game, however it has a dependency Player
class Game {
    private Player player;

    public Game(Player player) {
        this.player = player;
    }

    public String attack() {
        return "Player attack with: " + player.getWeapon();
    }
}

class Player {
    private String weapon;

    public Player(String weapon) {
        this.weapon = weapon;
    }

    String getWeapon() {
        return weapon;
    }
}

@RunWith(MockitoJUnitRunner.class) // Runner required to initialize @Mock and @InjectMocks
                                   // In JUnit 5, use @ExtendWith(MockitoExtension.class)
                                   // Or programmatically call MockitoAnnotations.openMocks(this)
class GameTest {
    // Similar to: player = mock(Player.class)
    @Mock
    Player player;

    // Mockito will inject the above created mock Player into Game
    // Note: Game itself is NOT a mock
    @InjectMocks
    Game game;

    @Test
    public void attackWithSwordTest() throws Exception {
        // Defining behavior of the mocked dependency
        Mockito.when(player.getWeapon()).thenReturn("Sword");

        assertEquals("Player attack with: Sword", game.attack());
    }
}